# Credit Card Fraud Dataset Analysis - Comparing Raw data performance with Statistically preprocessed data performance

## Step 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             precision_recall_curve)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

from statsmodels.stats.outliers_influence import variance_inflation_factor

import warnings
warnings.filterwarnings("ignore")

In [2]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

# print("Path to dataset files:", path)


In [3]:
# df = pd.read_csv(f"{path}/creditcard.csv")

## Step 2 — Load Dataset

In [4]:
df = pd.read_csv("creditcard.csv")

print(df.shape)
print(df["Class"].value_counts())

(284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64


## Step 3 — Train–Test Split (Stratified)

In [5]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

## Step 4 — Define Evaluation Function

In [6]:
def evaluate_model(model, X_train, y_train, X_test, y_test):

    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)[:,1]

    # threshold optimization
    precision, recall, thresholds = precision_recall_curve(y_test, probs)
    f1 = (2 * precision * recall) / (precision + recall + 1e-6)

    best_threshold = thresholds[np.argmax(f1)]

    preds = (probs >= best_threshold).astype(int)

    results = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds),
        "ROC_AUC": roc_auc_score(y_test, probs),
        "PR_AUC": average_precision_score(y_test, probs)
    }

    return results

## Step 5 — Define Raw Models

In [7]:
models_raw = {

    "Logistic": LogisticRegression(max_iter=200),

    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(n_neighbors=5),

    "XGBoost": XGBClassifier(
        eval_metric="logloss",
        random_state=42
    )
}

## Step 6 — Train Raw Models

In [8]:
results_raw = {}

for name, model in models_raw.items():

    res = evaluate_model(model, X_train, y_train, X_test, y_test)

    results_raw[name] = res

raw_df = pd.DataFrame(results_raw).T

print("RAW RESULTS")
print(raw_df)

RAW RESULTS
              Accuracy  Precision    Recall        F1   ROC_AUC    PR_AUC
Logistic      0.999228   0.787234  0.755102  0.770833  0.943095  0.675841
RandomForest  0.999649   0.933333  0.857143  0.893617  0.962348  0.873052
KNN           0.998420   0.833333  0.102041  0.181818  0.636393  0.114919
XGBoost       0.999508   0.926829  0.775510  0.844444  0.938952  0.797291


## 7. Statistical Preprocessing Pipeline
### 7.1 Feature Scaling

Necessary for logistic regression and KNN.

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

## Step 7.2 Remove Multicollinearity Using VIF

In [ ]:
def remove_high_vif(X, threshold=10):

    X = X.copy()

    while True:

        vif = pd.Series(
            [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
            index=X.columns
        )

        max_vif = vif.max()

        if max_vif > threshold:

            drop_feature = vif.idxmax()
            X = X.drop(columns=[drop_feature])

        else:
            break

    return X

X_train_vif = remove_high_vif(X_train_scaled)

X_test_vif = X_test_scaled[X_train_vif.columns]

## Step 7.3 SMOTE Oversampling

In [ ]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_vif, y_train)

print("After SMOTE:", np.bincount(y_train_smote))

After SMOTE: [227451 227451]


## Step 8. Define Statistically Enhanced Models

In [12]:
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

models_stat = {

    "Logistic": LogisticRegression(
        class_weight="balanced",
        max_iter=200
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ),

    "KNN": KNeighborsClassifier(n_neighbors=7),

    "XGBoost": XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42
    )
}

## Step 9 — Train Statistically Enhanced Models

In [13]:
results_stat = {}

for name, model in models_stat.items():

    res = evaluate_model(
        model,
        X_train_smote,
        y_train_smote,
        X_test_vif,
        y_test
    )

    results_stat[name] = res

stat_df = pd.DataFrame(results_stat).T

print("STATISTICAL RESULTS")
print(stat_df)

STATISTICAL RESULTS
              Accuracy  Precision    Recall        F1   ROC_AUC    PR_AUC
Logistic      0.999421   0.849462  0.806122  0.827225  0.970466  0.720335
RandomForest  0.999526   0.890110  0.826531  0.857143  0.980517  0.856753
KNN           0.999017   0.669355  0.846939  0.747748  0.948379  0.589291
XGBoost       0.999508   0.906977  0.795918  0.847826  0.981829  0.826812


## Step 10 — Compare Results

In [14]:
comparison = pd.concat(
    [raw_df, stat_df],
    axis=1,
    keys=["Raw", "Statistical"]
)

print(comparison)

                   Raw                                                    \
              Accuracy Precision    Recall        F1   ROC_AUC    PR_AUC   
Logistic      0.999228  0.787234  0.755102  0.770833  0.943095  0.675841   
RandomForest  0.999649  0.933333  0.857143  0.893617  0.962348  0.873052   
KNN           0.998420  0.833333  0.102041  0.181818  0.636393  0.114919   
XGBoost       0.999508  0.926829  0.775510  0.844444  0.938952  0.797291   

             Statistical                                                    
                Accuracy Precision    Recall        F1   ROC_AUC    PR_AUC  
Logistic        0.999421  0.849462  0.806122  0.827225  0.970466  0.720335  
RandomForest    0.999526  0.890110  0.826531  0.857143  0.980517  0.856753  
KNN             0.999017  0.669355  0.846939  0.747748  0.948379  0.589291  
XGBoost         0.999508  0.906977  0.795918  0.847826  0.981829  0.826812  


## Step 11 — Cross-Validated Evaluation

In [15]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = []

for train_idx, test_idx in skf.split(X, y):

    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    scaler = StandardScaler()

    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    smote = SMOTE(random_state=42)

    X_tr, y_tr = smote.fit_resample(X_tr, y_tr)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=200
    )

    model.fit(X_tr, y_tr)

    probs = model.predict_proba(X_te)[:,1]

    score = average_precision_score(y_te, probs)

    cv_scores.append(score)

print("Mean PR-AUC:", np.mean(cv_scores))

Mean PR-AUC: 0.728783324954375


# 2. Hybrid Model Implementation

## Step 1 — Import stacking tools

In [16]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

## Step 2. Define Base Models

In [17]:
base_models = [
    ("lr", LogisticRegression(class_weight="balanced", max_iter=200)),
    (
        "rf",
        RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=42
        ),
    ),
    ("knn", KNeighborsClassifier(n_neighbors=7)),
    (
        "xgb",
        XGBClassifier(
            scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42
        ),
    ),
]

## Step 3. Define Meta-Learner

In [18]:
meta_model = LogisticRegression()

## Step 4. Build Stacking Hybrid Model

In [19]:
hybrid_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1,
)

## Step 5. Train Hybrid Model

In [ ]:
# hybrid_model.fit(X_train_smote, y_train_smote)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('lr', ...), ('rf', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegression()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",5
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'predict_proba'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",-1
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0In

## Step 6. Saving and Loading hybrid model

In [ ]:
# saving the trained hybrid model
# import joblib
# joblib.dump(hybrid_model, "hybrid_model.joblib")

['hybrid_model.joblib']

In [27]:
# Load saved model
import joblib
hybrid_model = joblib.load("hybrid_model.joblib")

## Step 7. Evaluate Hybrid Model

In [28]:
probs = hybrid_model.predict_proba(X_test_vif)[:, 1]

In [29]:
precision, recall, thresholds = precision_recall_curve(y_test, probs)

f1_scores = (2 * precision * recall) / (precision + recall + 1e-6)

best_threshold = thresholds[np.argmax(f1_scores)]

preds = (probs >= best_threshold).astype(int)

In [32]:
results_hybrid = {
    "Accuracy": accuracy_score(y_test, preds),
    "Precision": precision_score(y_test, preds),
    "Recall": recall_score(y_test, preds),
    "F1": f1_score(y_test, preds),
    "ROC_AUC": roc_auc_score(y_test, probs),
    "PR_AUC": average_precision_score(y_test, probs),
}

print(results_hybrid)

{'Accuracy': 0.9995259997893332, 'Precision': 0.8817204301075269, 'Recall': 0.8367346938775511, 'F1': 0.8586387434554974, 'ROC_AUC': 0.9834634085767116, 'PR_AUC': 0.8629178646341392}


## Step 8. Compare All Models

In [36]:
all_results = pd.concat(
    [raw_df, stat_df, pd.DataFrame(results_hybrid, index=["Hybrid"])]
)
all_results.to_csv("all_results.csv")
print(all_results)

              Accuracy  Precision    Recall        F1   ROC_AUC    PR_AUC
Logistic      0.999228   0.787234  0.755102  0.770833  0.943095  0.675841
RandomForest  0.999649   0.933333  0.857143  0.893617  0.962348  0.873052
KNN           0.998420   0.833333  0.102041  0.181818  0.636393  0.114919
XGBoost       0.999508   0.926829  0.775510  0.844444  0.938952  0.797291
Logistic      0.999421   0.849462  0.806122  0.827225  0.970466  0.720335
RandomForest  0.999526   0.890110  0.826531  0.857143  0.980517  0.856753
KNN           0.999017   0.669355  0.846939  0.747748  0.948379  0.589291
XGBoost       0.999508   0.906977  0.795918  0.847826  0.981829  0.826812
Hybrid        0.999526   0.881720  0.836735  0.858639  0.983463  0.862918
